# 第19章　ハイパーパラメータ最適化と実験管理**『本格実装 医療診断支援AI（実装編）』のコード**本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**リポジトリ: https://github.com/kewel-corp/book-impl

## 手を動かす ― 探索を設計する

In [ ]:
import optunadef objective(trial):    lr  = trial.suggest_float("lr", 1e-5, 1e-2, log=True)   # 対数スケール    wd  = trial.suggest_float("wd", 1e-6, 1e-3, log=True)    aug = trial.suggest_categorical("aug", ["weak", "strong"])    model = train(lr=lr, wd=wd, aug=aug, epochs=15, report=trial)  # 途中経過を報告    if validate_fp_per_scan(model) > 2.0:  # 運用の必要条件を満たさない試行は不適格        raise optuna.TrialPruned()    return validate_ap(model)              # 検証データのAP（average precision）を最大化study = optuna.create_study(direction="maximize",                            pruner=optuna.pruners.MedianPruner())  # 中央値で枝刈りstudy.optimize(objective, n_trials=40)print(study.best_params)

## 二兎を追う ― 多目的最適化とPareto前線

In [ ]:
import optunadef objective(trial):    lr  = trial.suggest_float("lr", 1e-5, 1e-2, log=True)    beta= trial.suggest_float("tversky_beta", 0.5, 0.85)    model = train(lr=lr, beta=beta, epochs=15)    sens  = validate_sensitivity(model)        # 最大化したい    fp    = validate_fp_per_scan(model)         # 最小化したい    return sens, fpstudy = optuna.create_study(directions=["maximize", "minimize"],                            sampler=optuna.samplers.NSGAIISampler())study.optimize(objective, n_trials=60)for t in study.best_trials:                     # Pareto前線上の解たち    print(t.values, t.params)

## 平均を目的にし、ばらつきも一緒に見る

In [ ]:
import numpy as np, optunafrom sklearn.model_selection import StratifiedGroupKFolddef cv_score(params, n_splits=5, seed=0):    """一つのハイパーパラメータ組を、患者単位・層化のk-fold CVで評価する。"""    skf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)    scores = []    for tr, va in skf.split(X, y, groups=patient_ids):   # groups=患者ID（リーク防止）        model = train(X[tr], y[tr], **params)        scores.append(validate_dice(model, X[va], y[va]))    return float(np.mean(scores)), float(np.std(scores))def objective(trial):    params = {        "lr":   trial.suggest_float("lr", 1e-5, 1e-2, log=True),        "wd":   trial.suggest_float("wd", 1e-6, 1e-2, log=True),        "beta": trial.suggest_float("tversky_beta", 0.5, 0.85),    }    mean, std = cv_score(params)    trial.set_user_attr("std", std)          # ばらつきも記録しておく    return mean - 0.5 * std                  # 「平均が高く、foldで安定」を選ぶstudy = optuna.create_study(direction="maximize")study.optimize(objective, n_trials=30)